# Research Pipeline: Data Loading, Sampling, Preprocessing, and LIME Attribution
This notebook loads SNLI dataset, samples subsets, preprocesses them for attribution, and runs LIME explanations on SNLI examples.

Done for 300 samples of snli without stop words


## PART 1: Setup environment and data

### Import necessary libraries

In [31]:
import os
import json
import random
import torch
import numpy as np
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from lime.lime_text import LimeTextExplainer
from tqdm import tqdm
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import nltk

# Download NLTK stopwords if not already available
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
from nltk.corpus import stopwords

In [39]:
if torch.cuda.is_available(): # Check if CUDA is available
    print("Cuda is available")
    print("🔒 Clearing GPU memory...")
    torch.cuda.empty_cache()  # Clear GPU memory after execution
    torch.cuda.ipc_collect()  # Collect any remaining GPU memory
    print("🔒 GPU memory cleared after execution.")
    torch.cuda.synchronize()  # Ensure all operations are complete before exiting
    print("✅ Execution complete, GPU memory cleared.")
else:
    print("❌ Cuda is not available, skipping GPU memory cleanup.")

Cuda is available
🔒 Clearing GPU memory...
🔒 GPU memory cleared after execution.
✅ Execution complete, GPU memory cleared.


In [36]:
def _to_py(o):
    if isinstance(o, dict):
        return { _to_py(k): _to_py(v) for k, v in o.items() }
    if isinstance(o, (list, tuple)):
        return [ _to_py(v) for v in o ]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return o  # fall back

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def file_nonempty(path: str) -> bool:
    return os.path.isfile(path) and os.path.getsize(path) > 0

def read_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def write_json(path: str, data):
    ensure_dir(os.path.dirname(path))
    with open(path, "w", encoding="utf-8") as f:
        json.dump(_to_py(data), f, indent=2, ensure_ascii=False)


### Set Up Data Paths & Directories

In [4]:
def setup_directories():
    """Setup all required directories"""
    CWD = os.getcwd()
    DATA_DIR = os.path.join(CWD, "data")
    
    directories = {
        'DATA_DIR': DATA_DIR,
        'CACHE_DIR': os.path.join(DATA_DIR, "hf_cache"),
        'SNLI_LOCAL_DIR': os.path.join(DATA_DIR, "snli"),
        'SAMPLE_SNLI_DIR': os.path.join(DATA_DIR, "sampled_snli_data"),
        'PROCESSED_SNLI_DIR': os.path.join(DATA_DIR, "processed_snli_data"),
        'LIME_OUTPUT_DIR': os.path.join(DATA_DIR, "lime_outputs"),
    }
    
    # Create all directories
    for dir_path in directories.values():
        os.makedirs(dir_path, exist_ok=True)
    
    return directories

### Load Datasets and preview (with Local Cache)
Load SNLI dataset from disk if available, otherwise download and cache them locally.

In [5]:
def load_snli_dataset_fixed(dirs):
    """Load SNLI dataset with proper error handling"""
    print("📥 Loading SNLI dataset...")

    dataset_path = dirs['SNLI_LOCAL_DIR']
    dataset_ready = os.path.exists(os.path.join(dataset_path, "dataset_dict.json"))  # or "state.json"

    if dataset_ready:
        print("Loading from local cache...")
        snli_data = load_from_disk(dataset_path)
    else:
        print("Downloading SNLI dataset...")
        snli_data = load_dataset("snli", cache_dir=dirs['CACHE_DIR'])
        snli_data.save_to_disk(dataset_path)

    # Validate dataset
    if 'train' not in snli_data:
        raise ValueError("SNLI dataset does not contain 'train' split.")
    
    sample = snli_data['train'][0]
    required_fields = ['premise', 'hypothesis', 'label']
    for field in required_fields:
        if field not in sample:
            raise ValueError(f"SNLI dataset missing field: {field}")
    
    print(f"✅ SNLI dataset loaded: {len(snli_data['train'])} training examples")
    print("Sample:", sample)
    
    return snli_data


### Sample 300 Examples from Each Dataset and Save to disk
Randomly sample 300 valid examples from SNLI and CommonsenseQA for fast experimentation.

In [21]:
def sample_snli_dataset_fixed(dataset, dirs, num_samples=300):
    """Sample SNLI once; if sample exists, just load and return."""
    sample_json_path = os.path.join(dirs['SAMPLE_SNLI_DIR'], "snli_sample.json")

    if file_nonempty(sample_json_path):
        print(f"⏩ Found existing SNLI sample at {sample_json_path}; loading…")
        return read_json(sample_json_path)
    
    print(f"🎯 Sampling {num_samples} valid SNLI examples...")
    
    random.seed(42)
    
    # Filter out invalid labels (-1)
    valid_indices = [i for i, ex in enumerate(dataset['train']) if ex['label'] != -1]
    print(f"Found {len(valid_indices)} valid examples out of {len(dataset['train'])}")
    
    if len(valid_indices) < num_samples:
        raise ValueError(f"Not enough valid examples. Found {len(valid_indices)}, need {num_samples}")
    
    sample_indices = random.sample(valid_indices, num_samples)
    snli_sample_data = []
    
    LABEL_NAMES = ['entailment', 'neutral', 'contradiction']
    
    # Create samples with proper ID handling
    for i, idx in enumerate(sample_indices):
        ex = dataset['train'][idx]
        sample_entry = {
            'id': i,  # Use sequential ID since original might not have ID
            'premise': ex['premise'],
            'hypothesis': ex['hypothesis'],
            'label': ex['label'],
            'label_name': LABEL_NAMES[ex['label']]
        }
        snli_sample_data.append(sample_entry)
    
    # Save JSON
    write_json(sample_json_path, snli_sample_data)
    print(f"✅ Saved {len(snli_sample_data)} samples to: {sample_json_path}")
    
    return snli_sample_data

### Preprocess Sampled Data for Attribution
Convert SNLI and CSQA samples into model-ready format and save for later use in LIME/SHAP or other explainers.

In [7]:
def preprocess_snli_for_roberta(snli_sample_data, dirs):
    """
    Preprocess SNLI data specifically for RoBERTa-MNLI format
    Preprocess once; if processed file exists, load and return.
    """
    processed_path = os.path.join(dirs['PROCESSED_SNLI_DIR'], "processed_snli.json")
    if file_nonempty(processed_path):
        print(f"⏩ Found existing processed SNLI at {processed_path}; loading…")
        return read_json(processed_path)

    print("🔄 Preprocessing SNLI data for RoBERTa-MNLI...")
    
    processed_snli = []
    for ex in snli_sample_data:
        # RoBERTa format: premise</s></s>hypothesis  
        # But tokenizer handles this automatically, so we just use: premise<sep>hypothesis
        text = f"{ex['premise']}</s>{ex['hypothesis']}"
        
        processed_entry = {
            "id": ex["id"],
            "input_text": text,
            "premise": ex["premise"],
            "hypothesis": ex["hypothesis"],
            "label": ex["label"],
            "label_name": ex["label_name"],
            "dataset": "snli"
        }
        processed_snli.append(processed_entry)
    
    # Save processed data
    write_json(processed_path, processed_snli)
    
    print(f"✅ Saved {len(processed_snli)} processed examples to: {processed_path}")
    
    return processed_snli

## PART 2: ROBERTA-MNLI MODEL SETUP


In [ ]:
class RoBERTaMNLIClassifier:
    """Proper RoBERTa-MNLI classifier for LIME explanations"""
    
    def __init__(self, model_name="roberta-large-mnli"):
        print(f"🤖 Loading {model_name} model...")
        
        self.model_name = model_name
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")
        
        # Load model and tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        
        self.model.to(self.device)
        self.model.eval()
        
        # Label mapping for MNLI (RoBERTa uses different order than SNLI)
        self.label_mapping = {
            0: "contradiction",  # MNLI label 0
            1: "neutral",        # MNLI label 1  
            2: "entailment"      # MNLI label 2
        }
        
        print("✅ Model loaded successfully!")
        self._test_model()
    
    def _test_model(self):
        """Test model with a simple example"""
        print("🧪 Testing model...")
        
        test_premise = "The cat is sleeping on the couch."
        test_hypothesis = "The cat is awake."
        test_text = f"{test_premise}</s>{test_hypothesis}"
        
        probs = self.predict_proba([test_text])[0]
        predicted_label = int(np.argmax(probs))
        confidence = float(np.max(probs))
        
        print(f"Test input: '{test_premise}' vs '{test_hypothesis}'")
        print(f"Probabilities: {probs}")
        print(f"Predicted: {self.label_mapping[predicted_label]} (confidence: {confidence:.4f})")
        
        # Should predict contradiction with high confidence
        if predicted_label == 0 and confidence > 0.7:
            print("✅ Model test passed!")
        else:
            print("⚠️ Model test results seem unusual, but proceeding...")
    
    def predict_proba(self, texts):
        """Predict probabilities for LIME (batch processing)"""
        if isinstance(texts, str):
            texts = [texts]
        else:
            # normalize numpy/object arrays to a flat list of strings
            texts = [str(x) for x in np.array(texts, dtype=object).ravel().tolist()]

        premises, hyps = [], []
        for t in texts:
            if "</s>" in t:
                p, h = t.split("</s>", 1)
            else:
                # Fallback if LIME masks out the delimiter
                p, h = t, ""
            premises.append(p)
            hyps.append(h)

        inputs = self.tokenizer(
            premises,
            hyps,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1).detach().cpu().numpy()
        # Ensure 2D shape
        if probs.ndim == 1:
            probs = probs.reshape(1, -1)

        return probs

    def predict_single(self, text):
        """Get single prediction with details"""
        probs = self.predict_proba([text])[0]
        predicted_label = int(np.argmax(probs))
        
        return {
            'probabilities': probs,
            'predicted_label': predicted_label,
            'predicted_class': self.label_mapping[predicted_label],
            'confidence': float(np.max(probs)),
            'all_probs': {self.label_mapping[i]: probs[i] for i in range(len(probs))}
        }

## PART 3: LIME EXPLANATIONS WITH PROPER EVALUATION


In [38]:
def generate_lime_explanations(classifier, processed_snli, dirs, num_examples=50):
    """
    Generate LIME explanations with proper RoBERTa integration
    Generate LIME once; if file exists, load and return (no resume/overwrite).
    """
    lime_path = os.path.join(dirs['LIME_OUTPUT_DIR'], "lime_explanations_roberta.json")

     # If file exists but is empty or unreadable, we will regenerate
    if os.path.isfile(lime_path) and os.path.getsize(lime_path) > 0:
        try:
            data = read_json(lime_path)
            if isinstance(data, list) and len(data) > 0:
                print(f"⏩ Found existing LIME explanations at {lime_path}; loading…")
                return data[:num_examples] if len(data) >= num_examples else data
            else:
                print(f"⚠️ Found existing LIME file but it has 0 items. Will regenerate.")
        except Exception as e:
            print(f"⚠️ Failed to read existing LIME file ({e}). Will regenerate.")

    # --- inner helper that captures `classifier` ---
    def _lime_predict(batch, chunk_size=64):
        # normalize to list[str]
        if isinstance(batch, str):
            texts = [batch]
        else:
            texts = [str(x) for x in np.array(batch, dtype=object).ravel().tolist()]

        rows = []
        for i in range(0, len(texts), chunk_size):
            sub = texts[i:i+chunk_size]
            probs = classifier.predict_proba(sub)        # should return (len(sub), 3)
            probs = np.asarray(probs)
            if probs.ndim == 1:
                probs = probs.reshape(1, -1)
            if probs.shape[0] != len(sub):               # defensive fallback
                fixed = []
                for t in sub:
                    p = classifier.predict_proba([t])
                    p = np.asarray(p)
                    if p.ndim == 1:
                        p = p.reshape(1, -1)
                    fixed.append(p[0])
                probs = np.vstack(fixed)
            rows.append(probs)

        out = np.vstack(rows)
        assert out.shape[0] == len(texts), f"probs rows {out.shape[0]} != texts {len(texts)}"
        return out
    # -----------------------------------------------

    print(f"🔍 Generating LIME explanations for {num_examples} examples...")
    
    # Setup LIME explainer
    explainer = LimeTextExplainer(
        class_names=list(classifier.label_mapping.values()),
        verbose=False,
        random_state=42
    )
    
    lime_results = []
    
    for i, ex in enumerate(tqdm(processed_snli[:num_examples], desc="LIME Explanations")):
        try:
            text = ex["input_text"]
            
            # Get model prediction
            prediction = classifier.predict_single(text)
            
            # Generate LIME explanation
            explanation = explainer.explain_instance(
                text,
                _lime_predict,
                num_features=10,
                num_samples=1000,  # Increased for better quality
            )
            
            # Store results
            result = {
                "id": int(ex["id"]),
                "premise": ex["premise"],
                "hypothesis": ex["hypothesis"],
                "input_text": text,
                "true_label": int(ex["label"]),
                "true_label_name": ex["label_name"],
                "predicted_label": int(prediction["predicted_label"]),
                "predicted_class": prediction["predicted_class"],
                "confidence": float(prediction["confidence"]),
                "all_probabilities": {k: float(v) for k, v in prediction["all_probs"].items()},
                "lime_attributions": [(w, float(s)) for (w, s) in explanation.as_list()],
                "lime_score": float(explanation.score),
            }

            lime_results.append(result)
            
        except Exception as e:
            print(f"❌ Error processing example {i}: {e}")
            continue
    
    # Save LIME results
    write_json(lime_path, lime_results)
    
    print(f"✅ Generated {len(lime_results)} LIME explanations")
    print(f"✅ Saved to: {lime_path}")
    
    return lime_results

In [24]:
def filter_stopwords_from_lime(lime_results, dirs):
    """Remove stopwords from LIME attributions for cleaner analysis"""
    if not lime_results:
        print("⏩ No LIME results; skipping stopword filtering.")
        return []
    print("🧹 Filtering stopwords from LIME attributions...")
    
    # Setup stopwords
    stopwords_set = set(stopwords.words('english'))
    additional_stopwords = {"</s>", "[SEP]", "[CLS]", "[PAD]", "[UNK]", "[MASK]", "SEP"}
    stopwords_set.update(additional_stopwords)
    
    filtered_results = []
    for result in lime_results:
        # Filter attributions
        filtered_attributions = [
            (token, score) for token, score in result["lime_attributions"]
            if token.lower() not in stopwords_set and len(token.strip()) > 1
        ]
        
        # Create filtered result
        filtered_result = result.copy()
        filtered_result["lime_attributions_filtered"] = filtered_attributions
        filtered_result["original_attribution_count"] = len(result["lime_attributions"])
        filtered_result["filtered_attribution_count"] = len(filtered_attributions)
        
        filtered_results.append(filtered_result)
    
    # Save filtered results
    filtered_path = os.path.join(dirs['LIME_OUTPUT_DIR'], "lime_explanations_filtered.json")
    with open(filtered_path, "w", encoding="utf-8") as f:
        json.dump(filtered_results, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Filtered results saved to: {filtered_path}")
    print(f"Average attribution reduction: {np.mean([r['original_attribution_count'] - r['filtered_attribution_count'] for r in filtered_results]):.1f} tokens")
    
    return filtered_results

## PART 4: EVALUATION METRICS


In [11]:
def compute_faithfulness(classifier, original_text, top_tokens):
    """Compute faithfulness by removing top-k tokens"""
    try:
        # Remove top-k tokens
        words = original_text.split()
        perturbed_words = [w for w in words if w not in top_tokens]
        perturbed_text = " ".join(perturbed_words)
        
        if not perturbed_text.strip():
            return 0.0
        
        # Get predictions
        original_prob = classifier.predict_proba([original_text])[0].max()
        perturbed_prob = classifier.predict_proba([perturbed_text])[0].max()
        
        return abs(original_prob - perturbed_prob)
    
    except Exception:
        return 0.0

def compute_sufficiency(classifier, original_text, top_tokens):
    """Compute sufficiency by keeping only top-k tokens"""
    try:
        words = original_text.split()
        kept_words = [w for w in words if w in top_tokens]
        sufficient_text = " ".join(kept_words)
        
        if not sufficient_text.strip():
            return 0.0
        
        return classifier.predict_proba([sufficient_text])[0].max()
    
    except Exception:
        return 0.0

def compute_comprehensiveness(classifier, original_text, top_tokens):
    """Compute comprehensiveness"""
    try:
        words = original_text.split()
        remaining_words = [w for w in words if w not in top_tokens]
        remaining_text = " ".join(remaining_words)
        
        if not remaining_text.strip():
            return 0.0
        
        original_prob = classifier.predict_proba([original_text])[0].max()
        remaining_prob = classifier.predict_proba([remaining_text])[0].max()
        
        return original_prob - remaining_prob
    
    except Exception:
        return 0.0

In [12]:
def compute_evaluation_metrics(classifier, lime_results, k=3):
    """Compute faithfulness, sufficiency, and comprehensiveness"""
    print(f"📊 Computing evaluation metrics (k={k})...")
    
    metrics_results = []
    
    for result in tqdm(lime_results, desc="Computing metrics"):
        try:
            text = result["input_text"]
            attributions = result.get("lime_attributions_filtered", result["lime_attributions"])
            
            if len(attributions) < k:
                continue
            
            # Get top-k tokens
            sorted_attrs = sorted(attributions, key=lambda x: abs(x[1]), reverse=True)
            top_tokens = [token for token, _ in sorted_attrs[:k]]
            
            # Compute metrics
            faithfulness = compute_faithfulness(classifier, text, top_tokens)
            sufficiency = compute_sufficiency(classifier, text, top_tokens)
            comprehensiveness = compute_comprehensiveness(classifier, text, top_tokens)
            
            metric_result = {
                "id": result["id"],
                "faithfulness": faithfulness,
                "sufficiency": sufficiency,
                "comprehensiveness": comprehensiveness,
                "top_tokens": top_tokens,
                "k": k
            }
            
            metrics_results.append(metric_result)
            
        except Exception as e:
            print(f"❌ Error computing metrics for example {result['id']}: {e}")
            continue
    
    return metrics_results

In [43]:
import os
import numpy as np

def analyze_results(filtered_results, metrics_results, dirs):
    """Save a summary, accepting metrics_results as dict OR list OR (dict, list)."""
    if not filtered_results:
        print("⏩ No results; skipping analysis.")
        return {}

    def safe_float(x):
        try:
            return float(x)
        except Exception:
            return np.nan

    # 1) Normalize metrics_results into summary numbers
    avg_faithfulness = np.nan
    avg_sufficiency = np.nan
    avg_comprehensiveness = np.nan
    n = None
    per_example = None

    if isinstance(metrics_results, dict):
        # expected keys already there
        avg_faithfulness = safe_float(metrics_results.get("avg_faithfulness"))
        avg_sufficiency = safe_float(metrics_results.get("avg_sufficiency"))
        avg_comprehensiveness = safe_float(metrics_results.get("avg_comprehensiveness"))
        n = int(metrics_results.get("n", len(filtered_results)))
        per_example = metrics_results.get("per_example")

    elif isinstance(metrics_results, (list, tuple)):
        # tuple like (summary_dict, per_list)?
        if isinstance(metrics_results, tuple) and len(metrics_results) == 2 and isinstance(metrics_results[0], dict):
            summ, per = metrics_results
            avg_faithfulness = safe_float(summ.get("avg_faithfulness"))
            avg_sufficiency = safe_float(summ.get("avg_sufficiency"))
            avg_comprehensiveness = safe_float(summ.get("avg_comprehensiveness"))
            n = int(summ.get("n", len(filtered_results)))
            per_example = per
        else:
            # treat as list of per-example dicts
            per_example = list(metrics_results)
            n = len(per_example)
            if n:
                avg_faithfulness = float(np.nanmean([safe_float(x.get("faithfulness")) for x in per_example]))
                avg_sufficiency = float(np.nanmean([safe_float(x.get("sufficiency")) for x in per_example]))
                avg_comprehensiveness = float(np.nanmean([safe_float(x.get("comprehensiveness")) for x in per_example]))
            else:
                avg_faithfulness = avg_sufficiency = avg_comprehensiveness = float("nan")
    else:
        # unknown type; fall back to computing only from filtered_results
        n = len(filtered_results)

    # 2) Model performance from filtered_results (always available)
    total = len(filtered_results)
    correct = sum(1 for r in filtered_results if r.get("true_label_name") == r.get("predicted_class"))
    avg_confidence = float(np.mean([safe_float(r.get("confidence")) for r in filtered_results])) if total else float("nan")
    accuracy = float(correct / total) if total else 0.0

    # 3) Build final summary (cast to native Python types)
    final_results = {
        "evaluation": {
            "average_faithfulness": float(avg_faithfulness) if not np.isnan(avg_faithfulness) else None,
            "average_sufficiency": float(avg_sufficiency) if not np.isnan(avg_sufficiency) else None,
            "average_comprehensiveness": float(avg_comprehensiveness) if not np.isnan(avg_comprehensiveness) else None,
            "num_examples": int(n if n is not None else total),
        },
        "model_performance": {
            "accuracy": float(accuracy),
            "avg_confidence": float(avg_confidence),
            "correct": int(correct),
            "total": int(total),
        },
    }

    # (Optional) include per-example if you want it in the JSON
    # if per_example is not None:
    #     final_results["per_example"] = [
    #         {
    #             "id": int(r.get("id", i)),
    #             "faithfulness": safe_float(r.get("faithfulness")),
    #             "sufficiency": safe_float(r.get("sufficiency")),
    #             "comprehensiveness": safe_float(r.get("comprehensiveness")),
    #         } for i, r in enumerate(per_example)
    #     ]

    out_path = os.path.join(dirs["LIME_OUTPUT_DIR"], "complete_evaluation_results.json")
    write_json(out_path, final_results)  # uses your NumPy-safe writer
    print(f"✅ Complete results saved to: {out_path}")
    return final_results


## MAIN EXECUTION PIPELINE


In [44]:
def main():
    """Complete pipeline execution"""
    print("🎯 Starting Complete RoBERTa-MNLI + LIME Pipeline for SNLI")
    if torch.cuda.is_available(): # Check if CUDA is available
        print("Cuda is available")
        print("🔒 Clearing GPU memory...")
        torch.cuda.empty_cache()  # Clear GPU memory after execution
        torch.cuda.ipc_collect()  # Collect any remaining GPU memory
        print("🔒 GPU memory cleared after execution.")
        torch.cuda.synchronize()  # Ensure all operations are complete before exiting
        print("✅ Execution complete, GPU memory cleared.")
    else:
        print("❌ Cuda is not available, skipping GPU memory cleanup.")
    
    try:
        # Step 1: Setup
        dirs = setup_directories()
        print("✅ Directories setup complete")
        
        # Step 2: Load and sample data
        snli_data = load_snli_dataset_fixed(dirs)
        snli_sample = sample_snli_dataset_fixed(snli_data, dirs, num_samples=300)
        processed_snli = preprocess_snli_for_roberta(snli_sample, dirs)
        
        # Step 3: Load RoBERTa model
        classifier = RoBERTaMNLIClassifier("roberta-large-mnli")
        
        # Step 4: Generate LIME explanations
        lime_results = generate_lime_explanations(classifier, processed_snli, dirs, num_examples=50)
        
        # Step 5: Filter stopwords
        filtered_results = filter_stopwords_from_lime(lime_results, dirs)
        
        # Step 6: Evaluate
        metrics_results = compute_evaluation_metrics(classifier, filtered_results, k=3)
        
        # Step 7: Analyze and save results
        final_results = analyze_results(filtered_results, metrics_results, dirs)
        
        print("\n🎉 Pipeline completed successfully!")
        print("📁 Check the 'data' folder for all generated files")
        
        return final_results
        
    except Exception as e:
        print(f"❌ Pipeline failed: {e}")
        import traceback
        traceback.print_exc()
        return None

if __name__ == "__main__":
    results = main()

🎯 Starting Complete RoBERTa-MNLI + LIME Pipeline for SNLI
Cuda is available
🔒 Clearing GPU memory...
🔒 GPU memory cleared after execution.
✅ Execution complete, GPU memory cleared.
✅ Directories setup complete
📥 Loading SNLI dataset...
Loading from local cache...
✅ SNLI dataset loaded: 550152 training examples
Sample: {'premise': 'A person on a horse jumps over a broken down airplane.', 'hypothesis': 'A person is training his horse for a competition.', 'label': 1}
⏩ Found existing SNLI sample at c:\Users\Work\OneDrive\Desktop\ChatGPT\Research\project\data\sampled_snli_data\snli_sample.json; loading…
⏩ Found existing processed SNLI at c:\Users\Work\OneDrive\Desktop\ChatGPT\Research\project\data\processed_snli_data\processed_snli.json; loading…
🤖 Loading roberta-large-mnli model...
Using device: cuda


Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


✅ Model loaded successfully!
🧪 Testing model...
Test input: 'The cat is sleeping on the couch.' vs 'The cat is awake.'
Probabilities: [0.9956124  0.00212934 0.00225819]
Predicted: contradiction (confidence: 0.9956)
✅ Model test passed!
⏩ Found existing LIME explanations at c:\Users\Work\OneDrive\Desktop\ChatGPT\Research\project\data\lime_outputs\lime_explanations_roberta.json; loading…
🧹 Filtering stopwords from LIME attributions...
✅ Filtered results saved to: c:\Users\Work\OneDrive\Desktop\ChatGPT\Research\project\data\lime_outputs\lime_explanations_filtered.json
Average attribution reduction: 4.4 tokens
📊 Computing evaluation metrics (k=3)...


Computing metrics: 100%|██████████| 50/50 [00:06<00:00,  7.54it/s]

✅ Complete results saved to: c:\Users\Work\OneDrive\Desktop\ChatGPT\Research\project\data\lime_outputs\complete_evaluation_results.json

🎉 Pipeline completed successfully!
📁 Check the 'data' folder for all generated files
